# Gift Visuals — Free AI Video Generator (Colab GPU)

This notebook turns a free Colab GPU into a personal AI video generation server for the Gift Visuals app's **Generate (Beta)** feature.

**What this is:** an open-source model (AnimateDiff, with optional image conditioning) running on Google's free T4 GPU, exposed over the internet through a free ngrok tunnel.

**What this is NOT:** a production/always-on service. Free Colab sessions disconnect after a period of inactivity or after several hours. Generation is slow (often 1-5+ minutes per clip) and clips are short (a few seconds) and lower resolution — that is the real ceiling of what a free GPU can do, not a limitation of this notebook specifically.

**How to use it:**
1. In Colab: `Runtime` -> `Change runtime type` -> select **T4 GPU** -> Save.
2. Get a free ngrok authtoken at https://dashboard.ngrok.com/get-started/your-authtoken (free account, no card required) and paste it into the cell below.
3. Run every cell in order (top to bottom). The first run downloads several GB of model weights, so it takes a few minutes.
4. The last cell prints a **public URL** and an **access token** — copy both into the Gift Visuals app's Generate section.
5. Leave this tab open while you use the app. If Colab disconnects, just rerun everything and paste the new URL/token into the app again.

In [ ]:
# Step 1: install dependencies (torch + CUDA already ship with Colab, so we don't touch those).
!pip install -q diffusers transformers accelerate "imageio[ffmpeg]" flask flask-cors pyngrok pillow

In [ ]:
# Step 2: load the model. First run downloads ~4-6GB of weights - this can take several minutes.
import torch
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    print("WARNING: no GPU detected. Go to Runtime > Change runtime type > T4 GPU, then rerun this cell.")

BASE_MODEL = "emilianJR/epiCRealism"
MOTION_ADAPTER = "guoyww/animatediff-motion-adapter-v1-5-2"

adapter = MotionAdapter.from_pretrained(MOTION_ADAPTER, torch_dtype=torch.float16)
pipe = AnimateDiffPipeline.from_pretrained(BASE_MODEL, motion_adapter=adapter, torch_dtype=torch.float16)
pipe.scheduler = DDIMScheduler.from_pretrained(
    BASE_MODEL, subfolder="scheduler", clip_sample=False, timestep_spacing="linspace", beta_schedule="linear"
)

# Lets a reference image (uploaded in the app) steer the generated video's look,
# not just the text prompt alone.
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin")
pipe.set_ip_adapter_scale(0.6)

pipe.to(DEVICE)
pipe.enable_vae_slicing()
print("Model loaded on", DEVICE)

In [ ]:
# Step 3: the generation server. Runs a small Flask API with one real endpoint: /generate.
import os, io, base64, uuid, threading
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from PIL import Image
from diffusers.utils import export_to_video

app = Flask(__name__)
CORS(app)  # the Gift Visuals app calls this server directly from the browser

# A random token so a stranger who stumbles on your ngrok URL can't burn your free GPU quota.
# Set your own by defining GIFT_VISUALS_TOKEN before running this cell, or just use the random one printed below.
ACCESS_TOKEN = os.environ.get("GIFT_VISUALS_TOKEN") or uuid.uuid4().hex
print("=" * 60)
print("ACCESS TOKEN (paste this into the Gift Visuals app):")
print(ACCESS_TOKEN)
print("=" * 60)

OUTPUT_DIR = "/content/gift_visuals_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FPS = 8
MIN_DURATION_SECONDS = 1
MAX_DURATION_SECONDS = 4  # free-GPU reality check: longer clips get slow and VRAM-hungry fast

generation_lock = threading.Lock()  # a free T4 can only realistically do one generation at a time

def is_authorized(req):
    return req.headers.get("Authorization") == f"Bearer {ACCESS_TOKEN}"

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "device": DEVICE})

@app.route("/generate", methods=["POST"])
def generate():
    if not is_authorized(request):
        return jsonify({"error": "Unauthorized"}), 401

    data = request.get_json(force=True, silent=True) or {}
    prompt = (data.get("prompt") or "").strip()
    if not prompt:
        return jsonify({"error": "prompt is required"}), 400

    try:
        duration_seconds = float(data.get("durationSeconds", 2))
    except (TypeError, ValueError):
        duration_seconds = 2
    duration_seconds = min(max(duration_seconds, MIN_DURATION_SECONDS), MAX_DURATION_SECONDS)
    num_frames = max(8, int(duration_seconds * FPS))

    ip_adapter_image = None
    if data.get("imageBase64"):
        try:
            raw = base64.b64decode(data["imageBase64"].split(",")[-1])
            ip_adapter_image = Image.open(io.BytesIO(raw)).convert("RGB")
        except Exception as exc:
            return jsonify({"error": f"Could not read reference image: {exc}"}), 400

    if not generation_lock.acquire(blocking=False):
        return jsonify({"error": "Already generating a video — wait for it to finish and try again."}), 429

    try:
        kwargs = dict(
            prompt=prompt,
            negative_prompt="blurry, low quality, distorted, watermark, text",
            num_frames=num_frames,
            guidance_scale=7.5,
            num_inference_steps=25,
        )
        if ip_adapter_image is not None:
            kwargs["ip_adapter_image"] = ip_adapter_image

        result = pipe(**kwargs)
        frames = result.frames[0]

        job_id = uuid.uuid4().hex
        out_path = os.path.join(OUTPUT_DIR, f"{job_id}.mp4")
        export_to_video(frames, out_path, fps=FPS)

        return send_file(out_path, mimetype="video/mp4", as_attachment=True, download_name="ai-generated.mp4")
    except Exception as exc:
        return jsonify({"error": str(exc)}), 500
    finally:
        generation_lock.release()

def run_server():
    app.run(host="0.0.0.0", port=5000)

threading.Thread(target=run_server, daemon=True).start()
print("Server thread started on port 5000.")

In [ ]:
# Step 4: expose it publicly with ngrok (free tier).
# Get your own free authtoken at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = "PASTE_YOUR_FREE_NGROK_AUTHTOKEN_HERE"

from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_AUTHTOKEN
ngrok.kill()  # clear any stale tunnels from a previous run
public_url = ngrok.connect(5000, "http").public_url

print("=" * 60)
print("PASTE THIS INTO THE GIFT VISUALS APP'S 'GENERATION SERVER URL' FIELD:")
print(public_url)
print("=" * 60)
print("(and the ACCESS TOKEN printed in the previous cell into the 'Access token' field)")